# 5G用户预测
## 人工智能小组作业

**课程：** 人工智能
**题目：** 5G用户预测 - 二分类问题
**算法：** 随机森林（Random Forest）& XGBoost
**评估指标：** AUC（ROC曲线下面积）

---
### 问题背景
基于用户的个人信息和通信数据（话费、流量、活跃行为、套餐类型、区域等），预测用户是否为5G用户。

### 数据集说明
- 共60个字段，包含 `target`（预测目标）
- `cat_0 ~ cat_19`：20个离散型特征
- `num_0 ~ num_37`：38个数值型特征
- `id`：样本标识


## 1. 导入所需库


In [ ]:
import pandas as pd
import numpy as np
import os
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, roc_curve, auc
from sklearn.feature_selection import SelectKBest, f_classif
import xgboost as xgb
from imblearn.over_sampling import SMOTE

print("所有库导入成功！")


## 2. 数据加载与探索


In [ ]:
# 加载数据
script_dir = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."
train_csv_path = os.path.join(script_dir, "train.csv")
if not os.path.exists(train_csv_path):
    train_csv_path = "train.csv"

data = pd.read_csv(train_csv_path)
print(f"数据形状: {data.shape}")
print(f"列名: {list(data.columns)}")
print(f"目标变量分布:")
print(data["target"].value_counts())
print(f"正样本比例: {data['target'].mean():.4f}")


In [ ]:
# 识别特征类型
cat_cols = [col for col in data.columns if col.startswith("cat_")]
num_cols = [col for col in data.columns if col.startswith("num_")]
print(f"离散型特征数量: {len(cat_cols)}")
print(f"数值型特征数量: {len(num_cols)}")
print(f"总特征数: {len(cat_cols) + len(num_cols)}")

# 检查缺失值
print(f"总缺失值数: {data.isnull().sum().sum()}")

# 数值型特征基本统计
data[num_cols].describe()


In [ ]:
# 目标变量分布可视化
plt.figure(figsize=(8, 5))
colors = ["#2b6cb0", "#e53e3e"]
target_counts = data["target"].value_counts()
bars = plt.bar(["非5G用户 (0)", "5G用户 (1)"], target_counts.values, color=colors, edgecolor="white", width=0.6)
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(target_counts)*0.01,
             str(bar.get_height()), ha="center", va="bottom", fontsize=12, fontweight="bold")
plt.title("目标变量分布", fontsize=14, fontweight="bold")
plt.ylabel("数量")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## 3. 特征工程

### 3.1 离散型特征独热编码
将离散型特征转换为虚拟变量。

In [ ]:
# 对离散型特征进行独热编码
cat_cols = [col for col in data.columns if col.startswith("cat_")]
data_encoded = pd.get_dummies(data, columns=cat_cols, drop_first=True)
print(f"编码后特征数量: {data_encoded.shape[1]}")


### 3.2 数值型特征聚合
从数值列创建统计汇总特征。

In [ ]:
# 创建数值聚合特征
num_cols_encoded = [c for c in data_encoded.columns if c.startswith("num_")]
data_encoded["num_mean"] = data_encoded[num_cols_encoded].mean(axis=1)
data_encoded["num_std"] = data_encoded[num_cols_encoded].std(axis=1)
data_encoded["num_max"] = data_encoded[num_cols_encoded].max(axis=1)
data_encoded["num_min"] = data_encoded[num_cols_encoded].min(axis=1)
print("新增特征: num_mean, num_std, num_max, num_min")
print(f"特征工程后总特征数: {data_encoded.shape[1]}")


## 4. 特征选择
使用 SelectKBest 结合 f_classif 筛选出最重要的100个特征。

In [ ]:
# 划分特征和目标变量
X = data_encoded.drop(columns=["id", "target"])
y = data_encoded["target"]
print(f"特征矩阵形状: {X.shape}")

# 使用 SelectKBest 进行特征选择
n_samples = min(50000, len(X))
X_sample = X.sample(n=n_samples, random_state=42)
y_sample = y.loc[X_sample.index]

k = min(100, X.shape[1])
selector = SelectKBest(score_func=f_classif, k=k)
selector.fit(X_sample.fillna(0), y_sample)

# 获取选中的特征
selected_indices = selector.get_support(indices=True)
selected_features = [X.columns[i] for i in selected_indices]
X_selected = X.iloc[:, selected_indices]
print(f"从{X.shape[1]}个特征中选择了{len(selected_features)}个")

# 显示F值最高的10个特征
feature_scores = sorted(zip(selected_features, selector.scores_[selected_indices]),
                      key=lambda x: x[1], reverse=True)
print("F值最高的前10个特征:")
for i, (feat, score) in enumerate(feature_scores[:10]):
    print(f"  {i+1}. {feat}: {score:.2f}")


## 5. 数据预处理
标准化特征并使用SMOTE处理类别不平衡问题。

In [ ]:
# 标准化特征
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected.fillna(0))

# 划分训练集和测试集（分层抽样）
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"训练集: {X_train.shape[0]} 个样本")
print(f"  正样本比例: {y_train.mean():.4f}")
print(f"测试集: {X_test.shape[0]} 个样本")
print(f"  正样本比例: {y_test.mean():.4f}")

# 使用SMOTE处理类别不平衡
smote = SMOTE(random_state=42, sampling_strategy=0.5)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f"SMOTE过采样后 - 训练集: {X_train_res.shape[0]} 个样本")
print(f"SMOTE过采样后 - 正样本比例: {y_train_res.mean():.4f}")


## 6. 模型训练

### 6.1 随机森林 (Random Forest)
随机森林是一种集成学习方法，通过构建多棵决策树并对它们的预测结果取平均来提高准确性和稳定性。

In [ ]:
# 随机森林训练
print("正在训练随机森林...")
rf_start = time.time()

model_rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train_res, y_train_res)
rf_time = time.time() - rf_start
print(f"随机森林训练完成，耗时: {rf_time:.2f}秒")


### 6.2 XGBoost
XGBoost是一种梯度提升框架，以决策树为基学习器，并引入了正则化项防止过拟合。

In [ ]:
# XGBoost训练
print("正在训练XGBoost...")
xgb_start = time.time()

# 计算正负样本比例，用于XGBoost的scale_pos_weight参数
scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

model_xgb = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="auc",
    verbosity=0
)
model_xgb.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test, y_test)],
    verbose=False
)
xgb_time = time.time() - xgb_start
print(f"XGBoost训练完成，耗时: {xgb_time:.2f}秒")


## 7. 模型评估


In [ ]:
# 随机森林预测
y_pred_proba_rf = model_rf.predict_proba(X_test)[:, 1]
y_pred_rf = model_rf.predict(X_test)
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

# XGBoost预测
y_pred_proba_xgb = model_xgb.predict_proba(X_test)[:, 1]
y_pred_xgb = model_xgb.predict(X_test)
auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)

# 结果汇总
print("=" * 50)
print("结果汇总")
print("=" * 50)
print(f"{'模型':<20} {'AUC':<10} {'耗时':<10}")
print("-" * 40)
print(f"{'随机森林':<20} {auc_rf:<10.4f} {rf_time:<10.2f}秒")
print(f"{'XGBoost':<20} {auc_xgb:<10.4f} {xgb_time:<10.2f}秒")

# 分类报告
print("\n随机森林分类报告:")
print(classification_report(y_test, y_pred_rf, target_names=["非5G用户", "5G用户"]))
print("XGBoost分类报告:")
print(classification_report(y_test, y_pred_xgb, target_names=["非5G用户", "5G用户"]))


## 8. ROC曲线对比


In [ ]:
# 绘制ROC曲线
plt.figure(figsize=(10, 8))

# 随机森林ROC曲线
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)
plt.plot(fpr_rf, tpr_rf, color="#2b6cb0", lw=2.5,
         label=f"随机森林 (AUC = {auc_rf:.4f})")

# XGBoost ROC曲线
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_pred_proba_xgb)
plt.plot(fpr_xgb, tpr_xgb, color="#e53e3e", lw=2.5,
         label=f"XGBoost (AUC = {auc_xgb:.4f})")

# 对角线（随机分类器）
plt.plot([0, 1], [0, 1], color="gray", lw=1.5, linestyle="--", alpha=0.7,
         label="随机分类器 (AUC = 0.5)")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("假正率 (1 - 特异性)", fontsize=12)
plt.ylabel("真正率 (灵敏度)", fontsize=12)
plt.title("ROC曲线对比", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. 特征重要性分析


In [ ]:
# XGBoost特征重要性可视化
if hasattr(model_xgb, "feature_importances_"):
    importances = model_xgb.feature_importances_
    indices = np.argsort(importances)[::-1][:15]

    plt.figure(figsize=(10, 8))
    colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(indices)))[::-1]
    
    feature_names = [selected_features[i][:30] for i in indices]
    
    plt.barh(range(len(indices)), importances[indices][::-1], color=colors, edgecolor="white")
    plt.yticks(range(len(indices)), feature_names[::-1], fontsize=10)
    plt.xlabel("重要性分数", fontsize=12)
    plt.title("XGBoost 前15个重要特征", fontsize=14, fontweight="bold")
    plt.gca().invert_yaxis()
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("XGBoost 前15个重要特征:")
    for i, idx in enumerate(indices):
        feat_name = selected_features[idx][:35] if idx < len(selected_features) else f"特征_{idx}"
        print(f"  {i+1}. {feat_name:<35} {importances[idx]:.4f}")


In [ ]:
# 随机森林特征重要性可视化
if hasattr(model_rf, "feature_importances_"):
    rf_importances = model_rf.feature_importances_
    rf_indices = np.argsort(rf_importances)[::-1][:15]

    plt.figure(figsize=(10, 8))
    colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(rf_indices)))[::-1]
    
    feature_names = [selected_features[i][:30] for i in rf_indices]
    
    plt.barh(range(len(rf_indices)), rf_importances[rf_indices][::-1], color=colors, edgecolor="white")
    plt.yticks(range(len(rf_indices)), feature_names[::-1], fontsize=10)
    plt.xlabel("重要性分数", fontsize=12)
    plt.title("随机森林 前15个重要特征", fontsize=14, fontweight="bold")
    plt.gca().invert_yaxis()
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("随机森林 前15个重要特征:")
    for i, idx in enumerate(rf_indices):
        feat_name = selected_features[idx][:35] if idx < len(selected_features) else f"特征_{idx}"
        print(f"  {i+1}. {feat_name:<35} {rf_importances[idx]:.4f}")


## 10. 总结

### 总览
- **问题**：基于用户画像和通信数据的5G用户二分类预测
- **算法**：随机森林（Bagging）和 XGBoost（Boosting）
- **预处理**：独热编码、特征选择（SelectKBest）、标准化、SMOTE处理不平衡
- **评估指标**：AUC（ROC曲线下面积）

### 主要发现
- 两种集成方法均能有效预测5G用户
- XGBoost通常优于随机森林，得益于其Boosting机制和正则化
- 特征重要性分析揭示了哪些用户属性对预测最具影响力
- SMOTE过采样有助于提高模型在少数类上的表现

### 可改进方向
1. **超参数调优**：使用网格搜索或贝叶斯优化寻找最优参数
2. **高级特征工程**：创建离散型和数值型特征的交互项
3. **其他算法**：尝试LightGBM、CatBoost或神经网络
4. **集成堆叠**：组合多个模型以获得更好的性能
5. **阈值调优**：根据业务需求优化分类阈值
